In [5]:
import re
import json
from dataclasses import dataclass
from datetime import datetime
from typing import Optional, Dict, Any, List
import pandas as pd


@dataclass
class ParsedLogLine:
    timestamp: Optional[datetime]
    source: Optional[str]
    severity: Optional[str]
    message: str
    format_type: str

    def to_dict(self) -> Dict[str, Any]:
        return {
            "timestamp": self.timestamp,
            "source": self.source,
            "severity": self.severity,
            "message": self.message,
            "format_type": self.format_type,
        }


# ✅ COMPLETE TIMESTAMP_FORMATS
TIMESTAMP_FORMATS = [
    "%Y-%m-%d %H:%M:%S",
    "%Y-%m-%dT%H:%M:%S",
    "%Y-%m-%dT%H:%M:%S.%fZ",
    "%Y-%m-%dT%H:%M:%S.%f",
    "%d/%b/%Y:%H:%M:%S %z",  # Apache style
    "%b %d %H:%M:%S",        # Syslog
    "%H:%M:%S",              # Android bracket
]


def try_parse_timestamp(text: str) -> Optional[datetime]:
    if text is None:
        return None
    text = text.strip()
    if not text:
        return None
    for fmt in TIMESTAMP_FORMATS:
        try:
            return datetime.strptime(text, fmt)
        except Exception:
            continue
    return None


# Precompiled regex
SYSLOG_REGEX = re.compile(
    r"^(?P<timestamp>[A-Za-z]{3}\s+\d{1,2}\s+\d{2}:\d{2}:\d{2})\s+"
    r"(?P<hostname>[\w\-\.]+)\s+"
    r"(?P<process>[\w\-\[\]]+):\s+"
    r"(?P<message>.*)"
)

In [ ]:
# ─────────────────────────────────────────────────────────────
#  REGEX PATTERNS FOR ALL FORMATS
# ─────────────────────────────────────────────────────────────

PIPE_REGEX = re.compile(
    r"^(\d{2}/\w{3}/\d{4}:\d{2}:\d{2}:\d{2}\s+[+-]\d{2}:?\d{2})\s*\|\s*"
    r"(INFO|WARN|WARNING|ERROR|DEBUG|FATAL|CRITICAL|TRACE)\s*\|\s*"
    r"([a-zA-Z0-9_\-]+)\s*\|\s*(.*)",
    re.IGNORECASE
)

ISO_REGEX = re.compile(
    r"^(\d{4}-\d{2}-\d{2}T[\d:.]+(?:Z|[+-]\d{2}:?\d{2})?)\s+"
    r"(INFO|WARN|WARNING|ERROR|DEBUG|FATAL|CRITICAL|TRACE)\s+"
    r"([\w\-]+):?\s+(.*)",
    re.IGNORECASE
)

APP_REGEX = re.compile(
    r"^(\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2})\s+"
    r"(INFO|WARN|WARNING|ERROR|DEBUG|FATAL|CRITICAL|TRACE)\s+"
    r"\[([\w\-]+)\]\s+(.*)",
    re.IGNORECASE
)

CLF_REGEX = re.compile(
    r"^(\S+)\s+\S+\s+(\S+)\s+\[(\d{2}/\w{3}/\d{4}:\d{2}:\d{2}:\d{2}\s+[+-]\d{4})\]\s+"
    r'"(\S+)\s+(\S+)\s+\S+"\s+(\d{3})'
)

BRACKET_REGEX = re.compile(
    r"^\[(\d{2}:\d{2}:\d{2})\]\s+([IWED])/([\w\-]+):\s+(.*)"
)

KV_PAIRS_REGEX = re.compile(r"(\w+)=([^\s]+)")


# ─────────────────────────────────────────────────────────────
#  PARSERS FOR EACH FORMAT
# ─────────────────────────────────────────────────────────────

def parse_pipe_format(line: str) -> Optional[ParsedLogLine]:
    m = PIPE_REGEX.match(line)
    if m:
        ts_str, sev, src, msg = m.groups()
        return ParsedLogLine(
            timestamp=try_parse_timestamp(ts_str),
            source=src,
            severity=sev.lower(),
            message=msg,
            format_type="pipe"
        )
    return None


def parse_iso_format(line: str) -> Optional[ParsedLogLine]:
    m = ISO_REGEX.match(line)
    if m:
        ts_str, sev, src, msg = m.groups()
        return ParsedLogLine(
            timestamp=try_parse_timestamp(ts_str),
            source=src,
            severity=sev.lower(),
            message=msg,
            format_type="iso"
        )
    return None


def parse_app_format(line: str) -> Optional[ParsedLogLine]:
    m = APP_REGEX.match(line)
    if m:
        ts_str, sev, src, msg = m.groups()
        return ParsedLogLine(
            timestamp=try_parse_timestamp(ts_str),
            source=src,
            severity=sev.lower(),
            message=msg,
            format_type="app"
        )
    return None


def parse_syslog_format(line: str) -> Optional[ParsedLogLine]:
    m = SYSLOG_REGEX.match(line)
    if m:
        ts_str = m.group("timestamp")
        hostname = m.group("hostname")
        process = m.group("process")
        msg = m.group("message")
        return ParsedLogLine(
            timestamp=try_parse_timestamp(ts_str),
            source=hostname,
            severity=None,
            message=msg,
            format_type="syslog"
        )
    return None


def parse_bracket_format(line: str) -> Optional[ParsedLogLine]:
    m = BRACKET_REGEX.match(line)
    if m:
        ts_str, sev_char, src, msg = m.groups()
        sev_map = {"I": "info", "W": "warn", "E": "error", "D": "debug"}
        return ParsedLogLine(
            timestamp=try_parse_timestamp(ts_str),
            source=src,
            severity=sev_map.get(sev_char),
            message=msg,
            format_type="bracket"
        )
    return None


def parse_clf_format(line: str) -> Optional[ParsedLogLine]:
    m = CLF_REGEX.match(line)
    if m:
        client_ip, _, ts_str, method, path, status = m.groups()
        status_int = int(status)
        sev = "error" if status_int >= 500 else "warn" if status_int >= 400 else "info"
        return ParsedLogLine(
            timestamp=try_parse_timestamp(ts_str),
            source=client_ip,
            severity=sev,
            message=f"{method} {path} {status}",
            format_type="clf"
        )
    return None


def parse_json_format(line: str) -> Optional[ParsedLogLine]:
    if not (line.strip().startswith("{") and line.strip().endswith("}")):
        return None
    try:
        obj = json.loads(line)
        ts_str = obj.get("ts") or obj.get("timestamp") or obj.get("time")
        src = obj.get("svc") or obj.get("service") or obj.get("host")
        sev = obj.get("level") or obj.get("severity")
        msg = obj.get("msg") or obj.get("message") or ""
        return ParsedLogLine(
            timestamp=try_parse_timestamp(ts_str) if ts_str else None,
            source=src,
            severity=(sev.lower() if sev else None),
            message=str(msg),
            format_type="json"
        )
    except Exception:
        return None


def parse_kv_format(line: str) -> Optional[ParsedLogLine]:
    pairs = KV_PAIRS_REGEX.findall(line)
    if len(pairs) < 3:
        return None
    kv = dict(pairs)
    known_keys = {"ts", "time", "timestamp", "level", "sev", "msg", "message", "service", "svc", "host"}
    if len(set(kv.keys()) & known_keys) < 2:
        return None
    
    ts_str = kv.get("ts") or kv.get("time") or kv.get("timestamp")
    src = kv.get("service") or kv.get("svc") or kv.get("host")
    sev = kv.get("level") or kv.get("sev")
    msg = kv.get("msg") or kv.get("message") or line
    
    return ParsedLogLine(
        timestamp=try_parse_timestamp(ts_str) if ts_str else None,
        source=src,
        severity=(sev.lower() if sev else None),
        message=str(msg),
        format_type="kv"
    )


# ─────────────────────────────────────────────────────────────
#  MAIN PARSER CHAIN
# ─────────────────────────────────────────────────────────────

def parse_log_line(line: str) -> ParsedLogLine:
    """Try parsing with multiple format detectors."""
    if not line or not line.strip():
        return ParsedLogLine(None, None, None, "", "unknown")
    
    # Try each parser in order
    for parser in [
        parse_pipe_format,
        parse_json_format,
        parse_iso_format,
        parse_app_format,
        parse_syslog_format,
        parse_clf_format,
        parse_bracket_format,
        parse_kv_format,
    ]:
        result = parser(line)
        if result:
            return result
    
    # Fallback: unknown format
    return ParsedLogLine(None, None, None, line, "unknown")


def run_stage1_inmemory(filepath: str) -> pd.DataFrame:
    """Load and parse logs from file."""
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.read().splitlines()
    
    parsed_lines = [parse_log_line(line) for line in lines]
    records = [pl.to_dict() for pl in parsed_lines]
    
    return pd.DataFrame(records)


In [ ]:
# Test script - Stage 1 on your sample_logs.txt
print("🔄 Running Stage 1 on sample_logs.txt...")
df = run_stage1_inmemory("sample_logs.txt")

print(f"\n✅ Processed {len(df):,} lines")
print("\nFormat breakdown:")
print(df['format_type'].value_counts())

print("\nParsing success rates:")
unknown_pct = (df['format_type'] == 'unknown').mean() * 100
print(f"Unknown format: {unknown_pct:.1f}%")
print(f"Has timestamp: {(df['timestamp'].notna()).mean()*100:.1f}%")
print(f"Has severity: {(df['severity'].notna()).mean()*100:.1f}%")

print("\nFirst 10 parsed lines:")
print(df[['format_type', 'timestamp', 'severity', 'source', 'message']].head(10).to_string(index=False))

print("\nTop 5 unknown patterns (if any):")
unknowns = df[df.format_type == 'unknown']['message'].str[:60].value_counts().head()
if len(unknowns) > 0:
    print(unknowns)
else:
    print("No unknown patterns! 🎉")

ModuleNotFoundError: No module named 'stage1_ingest'